# M5 exploratory data analysis

What the M5 sales data looks like, and the modeling decisions that follow from it.

## Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch
from matplotlib.ticker import PercentFormatter

from demand_forecast.data.load import SALES_ID_COLUMNS, load_raw

plt.rcParams.update(
    {
        "figure.figsize": (11, 4),
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
pd.set_option("display.float_format", "{:,.3f}".format)

FIG_DIR = Path("../docs/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)


def save_figure(fig: plt.Figure, name: str) -> None:
    """Export a key figure for docs/eda.md."""
    fig.savefig(FIG_DIR / f"{name}.png", dpi=100, bbox_inches="tight")


calendar, prices, sales = load_raw(Path("../data/raw"))

In [ ]:
d_cols = [c for c in sales.columns if c.startswith("d_")]
meta = sales[SALES_ID_COLUMNS]
# The calendar runs 28 days past the last day of sales (the competition's forecast horizon).
days = calendar.iloc[: len(d_cols)].reset_index(drop=True)
assert days["d"].astype(str).tolist() == d_cols
dates = pd.DatetimeIndex(days["date"])
is_christmas = (dates.month == 12) & (dates.day == 25)
regular = ~is_christmas

# int32, not the loader's int16: daily totals across thousands of series overflow int16.
X = sales[d_cols].to_numpy(dtype=np.int32)

CATEGORIES = ["FOODS", "HOUSEHOLD", "HOBBIES"]
STATES = sorted(meta["state_id"].astype(str).unique())


def daily_by(*levels: str) -> pd.DataFrame:
    """Total units per day for each group: one row per date, one column per group."""
    keys = [meta[level] for level in levels]
    totals = pd.DataFrame(X).groupby(keys, observed=True).sum()
    return totals.T.set_axis(dates)


total = pd.Series(X.sum(axis=0), index=dates)
print(f"{X.shape[0]:,} series x {X.shape[1]:,} days, {X.nbytes / 1e6:.0f} MB")

## 1. Data overview

In [ ]:
pd.DataFrame(
    {
        "rows": [len(calendar), len(prices), len(sales)],
        "columns": [calendar.shape[1], prices.shape[1], sales.shape[1]],
        "MB": [
            df.memory_usage(deep=True).sum() / 1e6 for df in (calendar, prices, sales)
        ],
    },
    index=["calendar", "prices", "sales"],
)

In [ ]:
print(f"Sales:    {dates[0]:%Y-%m-%d} to {dates[-1]:%Y-%m-%d} ({len(dates):,} days)")
print(
    f"Calendar: {calendar['date'].min():%Y-%m-%d} to {calendar['date'].max():%Y-%m-%d}"
)
print(
    f"Prices:   Walmart weeks {prices['wm_yr_wk'].min()} to {prices['wm_yr_wk'].max()}"
)

In [ ]:
meta[["state_id", "store_id", "cat_id", "dept_id", "item_id"]].nunique().to_frame(
    "unique values"
)

In [ ]:
pd.crosstab(meta["store_id"], meta["cat_id"], margins=True)

In [ ]:
series_keys = pd.MultiIndex.from_frame(meta[["store_id", "item_id"]].astype(str))
price_keys = pd.MultiIndex.from_frame(
    prices[["store_id", "item_id"]].drop_duplicates().astype(str)
)

assert sales["id"].is_unique
assert series_keys.is_unique
assert (X >= 0).all()
assert not prices.duplicated(["store_id", "item_id", "wm_yr_wk"]).any()
assert (prices["sell_price"] > 0).all()
assert price_keys.isin(series_keys).all()
assert prices["wm_yr_wk"].isin(calendar["wm_yr_wk"]).all()
assert calendar["date"].diff().iloc[1:].eq(pd.Timedelta(days=1)).all()
print("All checks passed.")

**Findings.**
- 30,490 series: 3,049 items × 10 stores, in 3 states, 3 categories and 7 departments. Every item is listed in every store, so series differ only in when they launch.
- Sales cover 1,941 days (2011-01-29 to 2016-05-22). The calendar runs 28 days further; those days have no sales and are left out.
- All integrity checks pass: unique keys, no negative sales, positive prices, every price row maps to a series and a calendar week, no gaps in the calendar.

**Decision.** The raw files need no cleaning, deduplication or imputation before feature engineering.

## 2. Sales distribution

### Total sales over time

In [ ]:
ax = total.plot(lw=0.5, alpha=0.6, label="daily")
total.rolling(28, center=True).mean().plot(ax=ax, lw=2, label="28-day mean")
ax.set(title="Total units sold per day", xlabel="", ylabel="units")
ax.legend()
save_figure(ax.figure, "total_sales")
plt.show()

total[total < 0.1 * total.median()].to_frame("units")

### Share by level of the hierarchy

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, level in zip(axes, ["state_id", "store_id", "dept_id"]):
    share = daily_by(level).sum() / total.sum()
    share.sort_values().plot.barh(ax=ax)
    ax.set(title=f"Share of units by {level.removesuffix('_id')}", ylabel="")
    ax.xaxis.set_major_formatter(PercentFormatter(1))
fig.tight_layout()
plt.show()

In [ ]:
by_cat = daily_by("cat_id")[CATEGORIES]
ax = by_cat.rolling(28, center=True).mean().plot()
ax.set(title="Units per day by category (28-day mean)", xlabel="", ylabel="units")
plt.show()

**Findings.**
- Total sales grow from about 25,000 to 42,000 units a day over five years, with a dip around each turn of the year.
- Christmas Day is the only near-zero day (11 to 20 units in total): the stores are closed.
- Sales are concentrated. California has 44% of all units and its store CA_3 alone 17%. FOODS_3 is 49% of all units; HOBBIES_2 is under 1%.
- HOUSEHOLD roughly doubles over the period, HOBBIES is flat, and FOODS has the largest swings.

**Decision.**
- Store, state, category, department and item go into the model as categorical features: sales levels differ several-fold between them.
- Because of the trend, the model learns from recent history (lags, rolling means), and models are compared with scale-free metrics (WAPE, MASE, RMSSE).
- Christmas Day is dropped from training. It is a closure, not demand, and never falls in the 28-day test window.

## 3. Product launches

In [ ]:
first_price_week = (
    meta[["store_id", "item_id"]]
    .astype(str)
    .merge(
        prices.groupby(["store_id", "item_id"], observed=True)["wm_yr_wk"]
        .min()
        .rename("first_price_week")
        .reset_index()
        .astype({"store_id": str, "item_id": str}),
        how="left",
    )["first_price_week"]
)
# An item is on the shelf from the first day of its first week with a price.
launch_day = np.searchsorted(days["wm_yr_wk"].to_numpy(), first_price_week.to_numpy())
on_shelf = np.arange(len(dates)) >= launch_day[:, None]
sold = X > 0
first_sale_day = np.where(sold.any(axis=1), sold.argmax(axis=1), len(dates))
launched_late = launch_day > 0

print(f"Series on the shelf from day 1:  {1 - launched_late.mean():.1%}")
print(f"Series-days before launch:       {1 - on_shelf.mean():.1%}")
print(f"Zeros that are pre-launch days:  {(~on_shelf).sum() / (X == 0).sum():.1%}")
print(f"Units sold before launch:        {X[~on_shelf].sum():,}")

In [ ]:
launches = pd.DataFrame(
    {
        "quarter": dates[np.minimum(launch_day, len(dates) - 1)].to_period("Q"),
        "cat_id": meta["cat_id"].astype(str),
    }
)[launched_late]
ax = (
    launches.groupby(["quarter", "cat_id"])
    .size()
    .unstack()
    .reindex(columns=CATEGORIES)
    .plot.bar(stacked=True, figsize=(13, 4))
)
ax.set(
    title="Series launched after the start of the data, by quarter",
    xlabel="",
    ylabel="series",
)
save_figure(ax.figure, "launches")
plt.show()

In [ ]:
gap = pd.Series(first_sale_day - launch_day, name="days from first price to first sale")
gap.describe(percentiles=[0.5, 0.9, 0.99]).to_frame()

**Findings.**
- Only 36% of series are on the shelf from the first day. The rest launch over the next five years, mostly in 2011 to 2014.
- 21% of all series-days fall before a series' first price, and they make up 31% of all zero-sales days.
- Nothing sells before the first price week, and the first sale follows it within 2 days at the median (6 days at the 99th percentile). The first price week is a reliable launch date.

**Decision.** Days before a series' first price week are removed from features, training and metric scaling. Counting them as zero demand would push forecasts for new items down. This matches the official RMSSE, whose scale starts at a series' first sale.

## 4. Intermittency

In [ ]:
n_active = on_shelf.sum(axis=1)
n_sold = sold.sum(axis=1)
units_sum = X.sum(axis=1)
units_sq = np.einsum("ij,ij->i", X, X, dtype=np.int64)
size_mean = units_sum / n_sold

stats = (
    meta[["cat_id", "store_id"]]
    .astype(str)
    .assign(
        active_days=n_active,
        mean_units=units_sum / n_active,
        zero_share=1 - n_sold / n_active,
        adi=n_active / n_sold,
        cv2=(units_sq / n_sold - size_mean**2) / size_mean**2,
    )
)
active_zero = on_shelf & ~sold
print(f"Zero share, all series-days:    {(X == 0).mean():.1%}")
print(f"Zero share, on-shelf days only: {active_zero.sum() / on_shelf.sum():.1%}")
stats.groupby("cat_id")[["mean_units", "zero_share"]].median()

### Sales velocity and share of zero days per series

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
velocity_bins = np.logspace(-2, 3, 60)
for cat in CATEGORIES:
    in_cat = stats["cat_id"] == cat
    axes[0].hist(
        stats.loc[in_cat, "mean_units"],
        bins=velocity_bins,
        histtype="step",
        lw=2,
        label=cat,
    )
    axes[1].hist(
        stats.loc[in_cat, "zero_share"],
        bins=50,
        range=(0, 1),
        histtype="step",
        lw=2,
        label=cat,
    )
axes[0].set(
    xscale="log",
    title="Mean units per on-shelf day",
    xlabel="units (log)",
    ylabel="series",
)
axes[1].set(
    title="Share of on-shelf days with zero sales", xlabel="zero share", ylabel="series"
)
axes[1].xaxis.set_major_formatter(PercentFormatter(1))
axes[0].legend()
fig.tight_layout()
save_figure(fig, "intermittency")
plt.show()

### Demand classes (Syntetos-Boylan)

In [ ]:
ADI_CUT, CV2_CUT = 1.32, 0.49
DEMAND_CLASSES = ["smooth", "erratic", "intermittent", "lumpy"]
frequent = stats["adi"] < ADI_CUT
stable = stats["cv2"] < CV2_CUT
stats["demand_class"] = np.select(
    [frequent & stable, frequent & ~stable, ~frequent & stable],
    DEMAND_CLASSES[:3],
    default="lumpy",
)
pd.crosstab(
    stats["cat_id"], stats["demand_class"], normalize="index", margins=True
).reindex(columns=DEMAND_CLASSES, fill_value=0)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sample = stats.sample(min(len(stats), 5000), random_state=0)
for name in DEMAND_CLASSES:
    part = sample[sample["demand_class"] == name]
    ax.scatter(part["adi"], part["cv2"], s=4, alpha=0.4, label=name)
ax.axvline(ADI_CUT, color="grey", ls="--")
ax.axhline(CV2_CUT, color="grey", ls="--")
ax.set(
    xscale="log",
    yscale="log",
    title="Demand classes (5,000 random series)",
    xlabel="ADI: mean days between sales (log)",
    ylabel="CV²: variability of non-zero sales (log)",
)
ax.legend(markerscale=4)
plt.show()

In [ ]:
fig, axes = plt.subplots(len(DEMAND_CLASSES), 1, figsize=(13, 9), sharex=True)
for ax, name in zip(axes, DEMAND_CLASSES):
    candidates = stats.index[stats["demand_class"] == name]
    if candidates.empty:
        continue
    i = pd.Series(candidates).sample(1, random_state=0).iloc[0]
    ax.plot(dates[-365:], X[i, -365:], lw=1)
    ax.set(title=f"{name}: {sales['id'].iloc[i]}", ylabel="units")
fig.suptitle("Last year of one random series per demand class")
fig.tight_layout()
plt.show()

### Long runs of zeros

In [ ]:
# Length of the current run of zero-sale on-shelf days; any sale or pre-launch day resets it.
day_idx = np.arange(len(dates), dtype=np.int32)
last_reset = np.maximum.accumulate(np.where(active_zero, -1, day_idx), axis=1)
stats["max_zero_run"] = (day_idx - last_reset).max(axis=1)
stats["trailing_zero_run"] = day_idx[-1] - last_reset[:, -1]
del last_reset

runs = (
    pd.DataFrame(
        {
            f"{label} > {n} days": stats[column] > n
            for column, label in [
                ("max_zero_run", "longest run"),
                ("trailing_zero_run", "run at end"),
            ]
            for n in (28, 91)
        }
    )
    .groupby(stats["cat_id"])
    .mean()
)
runs.style.format("{:.1%}")

**Findings.**
- 60% of on-shelf days have zero sales (68% before removing pre-launch days). The median series sells 0.93 units a day in FOODS, 0.44 in HOUSEHOLD and 0.31 in HOBBIES.
- Sales rates span more than three orders of magnitude, from under 0.1 to over 100 units a day.
- 73% of series are intermittent and 18% lumpy; only 6% are smooth. HOUSEHOLD and HOBBIES are almost all intermittent, while FOODS has the most lumpy and smooth series.
- 79% of series have a run of more than 28 days without a sale, and 48% one of more than 91 days. With this many zero days, such runs also happen by chance, so they cannot be read as stock-outs from sales alone.
- 2% to 3% of series end with more than 28 days without a sale and may have been delisted.

**Decision.**
- No MAPE or other per-series percentage error: it is undefined on zero days. WAPE, MASE and RMSSE as planned.
- LightGBM uses a Tweedie objective instead of squared error, since the target is a non-negative count with a large mass at zero.
- A moving-average baseline joins the naive and seasonal-naive baselines: on intermittent series, last-value forecasts are mostly zero.
- "Days since last sale" becomes a feature; demand class and sales rate become slices for the error analysis.

## 5. Seasonality

### Day of week

In [ ]:
WEEKDAYS = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
]
fig, axes = plt.subplots(1, 2, figsize=(15, 4), sharey=True)
for ax, level, order in zip(axes, ["cat_id", "state_id"], [CATEGORIES, STATES]):
    daily = daily_by(level)[order][regular]
    profile = daily.groupby(daily.index.day_name()).mean().reindex(WEEKDAYS)
    (profile / profile.mean()).plot(ax=ax, marker="o")
    ax.set(title=f"Weekly profile by {level.removesuffix('_id')}", xlabel="")
axes[0].set_ylabel("mean units / weekly mean")
fig.tight_layout()
save_figure(fig, "weekly_profile")
plt.show()

### Day of month

In [ ]:
cat_daily = by_cat[regular]
profile = cat_daily.groupby(cat_daily.index.day).mean()
ax = (profile / profile.mean()).plot(marker=".")
ax.set(
    title="Profile by day of month",
    xlabel="day of month",
    ylabel="mean units / monthly mean",
)
plt.show()

### Month of year and long-term trend

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
regular_total = total[regular]
by_year = (
    regular_total.groupby([regular_total.index.year, regular_total.index.month])
    .mean()
    .unstack(0)
)
by_year.plot(ax=axes[0], marker=".")
axes[0].set(
    title="Mean units per day by month, one line per year",
    xlabel="month",
    ylabel="units",
)
axes[0].legend(title="year")

# Divide by a centered 365-day mean so the growth trend does not leak into the month profile.
detrended = (by_cat / by_cat.rolling(365, center=True, min_periods=300).mean())[regular]
month_profile = detrended.groupby(detrended.index.month).mean()
month_profile.plot(ax=axes[1], marker="o")
axes[1].axhline(1, color="grey", ls="--")
axes[1].set(
    title="Month profile by category, trend removed",
    xlabel="month",
    ylabel="units / 365-day mean",
)
fig.tight_layout()
plt.show()

**Findings.**
- Every category and state has a strong weekly cycle: Saturday and Sunday sell 12% to 26% above the weekly mean, Tuesday to Thursday 10% to 15% below. HOBBIES and Wisconsin drop back on Sunday.
- FOODS sells up to 14% above average in the first half of the month and about 10% below in the last week. HOUSEHOLD and HOBBIES are flat apart from the first three days. The FOODS pattern lines up with SNAP days (section 7).
- Month of year matters much less than day of week: with the trend removed, months move sales by at most 10%. FOODS and HOUSEHOLD peak in summer (June to September, up to 8% above trend) and are lowest from November to January. HOBBIES has no clear yearly pattern.

**Decision.**
- Day of week, day of month and month are calendar features. Lags and rolling windows are multiples of 7 days so they keep the weekly phase.
- The seasonal-naive baseline uses a 7-day season.

## 6. Events and holidays

In [ ]:
events = (
    pd.concat(
        [
            days[["date", f"event_name_{i}", f"event_type_{i}"]].set_axis(
                ["date", "event", "type"], axis=1
            )
            for i in (1, 2)
        ]
    )
    .dropna(subset=["event"])
    .astype({"event": str, "type": str})
    .reset_index(drop=True)
)
event_dates = pd.DatetimeIndex(events["date"])


def lift(series: pd.Series, at: pd.DatetimeIndex) -> pd.Series:
    """Units on each date relative to the same weekday 1-4 weeks before and after."""
    clean = series[~series.index.isin(event_dates)]
    offsets = [pd.Timedelta(weeks=k) for k in (-4, -3, -2, -1, 1, 2, 3, 4)]
    baseline = pd.concat(
        [clean.reindex(at + o).set_axis(at) for o in offsets], axis=1
    ).mean(axis=1)
    return series.reindex(at) / baseline


events["lift"] = lift(total, event_dates).to_numpy()
for cat in CATEGORIES:
    events[cat] = lift(by_cat[cat], event_dates).to_numpy()
print(f"{len(events)} event days, {events['event'].nunique()} distinct events")
events.groupby("type")[["lift", *CATEGORIES]].mean()

In [ ]:
per_event = events.groupby("event")["lift"].mean().sort_values()
ax = per_event.plot.barh(figsize=(8, 9))
ax.axvline(1, color="grey", ls="--")
ax.set(
    title="Mean lift on the event day (1 = normal day)",
    xlabel="units / same weekday nearby",
    ylabel="",
)
plt.show()

### Days around key events

In [ ]:
KEY_EVENTS = [
    "SuperBowl",
    "Easter",
    "Mother's day",
    "IndependenceDay",
    "Thanksgiving",
    "NewYear",
]
window = range(-7, 4)
total_open = total.where(
    regular
)  # Christmas closures would otherwise show up as dips around other events
around = pd.DataFrame(
    {
        name: [
            lift(
                total_open,
                pd.DatetimeIndex(events.loc[events["event"] == name, "date"])
                + pd.Timedelta(days=k),
            ).mean()
            for k in window
        ]
        for name in KEY_EVENTS
        if (events["event"] == name).any()
    },
    index=window,
)
ax = around.plot(marker=".")
ax.axhline(1, color="grey", ls="--")
ax.set(
    title="Lift in the days around key events", xlabel="days from event", ylabel="lift"
)
save_figure(ax.figure, "events")
plt.show()

total[is_christmas].to_frame("units on Christmas")

**Findings.**
- Most events change sales by less than 5% on the day.
- The large holidays lower sales on the day itself: Thanksgiving by 30%, New Year by 20%, Easter and Mother's Day by about 14%. Labor Day is the exception, at 26% above a normal day.
- Shopping moves to the days before: sales are 10% to 28% higher one or two days ahead of Independence Day, Thanksgiving, Easter, the Super Bowl and New Year.
- By event type the effects cancel out (Religious and Sporting average 1.00), so the type alone carries little signal.

**Decision.**
- Events are encoded by name, not only by type, together with "days until the next event" (capped at a week) to capture shopping ahead of holidays.
- Event days versus normal days is a slice for the error analysis.

## 7. SNAP effect

In [ ]:
state_cat = daily_by("state_id", "cat_id")[regular]
regular_days = days[regular]
first_half = (regular_days["date"].dt.day <= 15).to_numpy()
rows = []
for state in STATES:
    snap = (regular_days[f"snap_{state}"] == 1).to_numpy()
    for cat in CATEGORIES:
        s = state_cat[(state, cat)].to_numpy()
        rows.append(
            {
                "state": state,
                "category": cat,
                "all days": s[snap].mean() / s[~snap].mean(),
                "days 1-15 only": s[snap & first_half].mean()
                / s[~snap & first_half].mean(),
            }
        )
pd.DataFrame(rows).pivot(index="category", columns="state").reindex(CATEGORIES)

In [ ]:
fig, axes = plt.subplots(1, len(STATES), figsize=(15, 4), sharey=True)
for ax, state in zip(axes, STATES):
    foods = state_cat[(state, "FOODS")]
    by_day = foods.groupby(foods.index.day).mean()
    snap_share = (
        regular_days[f"snap_{state}"].groupby(regular_days["date"].dt.day).mean()
    )
    ax.bar(
        by_day.index,
        by_day / by_day.mean(),
        color=np.where(snap_share > 0.5, "C1", "C0"),
    )
    ax.set(title=f"{state}: FOODS by day of month", xlabel="day of month")
axes[0].set_ylabel("mean units / overall mean")
axes[-1].legend(
    handles=[Patch(color="C1", label="SNAP day"), Patch(color="C0", label="no SNAP")]
)
fig.tight_layout()
save_figure(fig, "snap_foods")
plt.show()

**Findings.**
- On SNAP days, FOODS sells 10% more in California, 15% in Texas and 30% in Wisconsin than on other days. HOUSEHOLD and HOBBIES gain only 1% to 3%.
- Part of this is the start-of-month effect. Comparing only days 1 to 15, the FOODS lift shrinks to 6%, 12% and 20%, and HOUSEHOLD and HOBBIES drop to about zero.
- Wisconsin shows it most clearly: SNAP and non-SNAP days alternate in the first half of the month, and FOODS sales follow them day by day.

**Decision.**
- Each series gets the SNAP flag of its own state, as a single `snap` feature rather than three.
- Day of month stays a separate feature, so the model can tell SNAP apart from the start-of-month effect.

## 8. Prices

### How often prices change

In [ ]:
by_series = prices.sort_values(["store_id", "item_id", "wm_yr_wk"]).groupby(
    ["store_id", "item_id"], observed=True
)["sell_price"]
price_stats = pd.DataFrame(
    {
        "weeks": by_series.size(),
        "changes": by_series.agg(lambda p: (p.diff().abs() > 0.005).sum()),
        "distinct_prices": by_series.nunique(),
    }
).reset_index()
price_stats["cat_id"] = price_stats["item_id"].astype(str).str.split("_").str[0]
price_stats["changes_per_year"] = price_stats["changes"] / price_stats["weeks"] * 52
price_stats.groupby("cat_id")[["changes_per_year", "distinct_prices"]].describe(
    percentiles=[0.5, 0.9]
).T.loc[(slice(None), ["mean", "50%", "90%"]), :]

### Same item, different stores

In [ ]:
spread = prices.groupby(["item_id", "wm_yr_wk"], observed=True)["sell_price"].agg(
    ["min", "max"]
)
spread["ratio"] = spread["max"] / spread["min"]
spread["cat_id"] = (
    spread.index.get_level_values("item_id").astype(str).str.split("_").str[0]
)
spread.groupby("cat_id")["ratio"].agg(
    same_price_everywhere=lambda r: (r < 1.005).mean(),
    median_max_to_min="median",
    p90_max_to_min=lambda r: r.quantile(0.9),
)

### Price and sales

In [ ]:
week = days["wm_yr_wk"].to_numpy()
starts = np.flatnonzero(np.r_[True, week[1:] != week[:-1]])
full = np.diff(np.r_[starts, len(week)]) == 7
weekly = np.add.reduceat(X, starts, axis=1)[:, full]
week_ids = week[starts][full]

units_week = pd.DataFrame(
    {
        "series": np.repeat(np.arange(len(sales)), len(week_ids)),
        "wm_yr_wk": np.tile(week_ids, len(sales)),
        "units": weekly.ravel(),
    }
)
series_ids = meta[["store_id", "item_id", "cat_id"]].assign(
    series=np.arange(len(sales))
)
series_ids = series_ids.astype(
    {"store_id": prices["store_id"].dtype, "item_id": prices["item_id"].dtype}
)
pw = prices.merge(series_ids, on=["store_id", "item_id"]).merge(
    units_week, on=["series", "wm_yr_wk"]
)
pw["rel_price"] = pw["sell_price"] / pw.groupby("series")["sell_price"].transform(
    "median"
)
pw["rel_units"] = pw["units"] / pw.groupby("series")["units"].transform("mean")
pw = pw[pw["rel_units"].notna()]

PRICE_BANDS = ["< 0.80", "0.80-0.90", "0.90-0.97", "0.97-1.03", "1.03-1.10", "> 1.10"]
pw["price_band"] = pd.cut(
    pw["rel_price"], [0, 0.8, 0.9, 0.97, 1.03, 1.1, np.inf], labels=PRICE_BANDS
)
print("Share of series-weeks per price band (price / series median):")
print(
    pw["price_band"]
    .value_counts(normalize=True)
    .reindex(PRICE_BANDS)
    .map("{:.1%}".format)
    .to_string()
)

ax = (
    pw.groupby(["price_band", "cat_id"], observed=True)["rel_units"]
    .mean()
    .unstack()[CATEGORIES]
    .plot.bar(rot=0)
)
ax.axhline(1, color="grey", ls="--")
ax.set(
    title="Weekly units vs price, relative to each series' own median price and mean sales",
    xlabel="price / series median price",
    ylabel="units / series mean",
)
plt.show()

**Findings.**
- Prices rarely change. The median series changes price 0.55 times a year in FOODS and about 0.3 times in HOUSEHOLD and HOBBIES. 79% of series-weeks are within 3% of the series' median price.
- Prices are mostly the same across stores: in 64% to 82% of item-weeks every store charges the same, and the 90th percentile gap between the cheapest and the most expensive store is 7% to 23%.
- Weeks priced below a series' median sell 13% to 30% more than usual, and weeks priced above it 5% to 16% less, in all three categories. This is a correlation, not an elasticity: price changes also coincide with trend and launch effects.

**Decision.**
- Price features are relative to the series' own history (price over its recent maximum or median, and week-over-week change). Cross-store price comparisons are skipped, since prices rarely differ between stores.
- Forecasting needs the price for the forecast days, so the forecasting API takes planned prices as input.

## 9. Summary

| Finding | Decision |
|---|---|
| 64% of series launch after the first day; pre-launch days are 31% of all zeros | Drop pre-launch days from features, training and metric scaling |
| Christmas Day is a store closure | Drop it from training |
| Strong weekly cycle; FOODS peaks in the first half of the month | Day of week, day of month and month features; windows in multiples of 7 days |
| SNAP lifts FOODS by 6% to 20% beyond the start-of-month effect | A `snap` feature from each series' own state |
| Big holidays lower sales on the day and raise them on the days before | Event name and "days until next event" features |
| Prices rarely change and are mostly uniform across stores; cheaper weeks sell more | Price relative to the series' own history |
| 60% of on-shelf days are zero; 73% of series are intermittent | WAPE, MASE, RMSSE (no MAPE); Tweedie objective; moving-average baseline |
| Sales levels differ several-fold across stores, departments and items | Hierarchy as categorical features; scale-free metrics |
| Demand class, sales rate and event days behave differently | Slices for the error analysis |